# Code mẫu — Object detection

Một model detection trả về một danh sách box, mỗi box kèm lớp và điểm tin cậy. Các ví dụ dưới đây đi theo đúng thứ tự xử lý: đo độ chồng giữa hai box, bỏ các box trùng lặp, rồi ghép với nhãn thật để chấm điểm, sao cho mỗi vật thể chỉ được tính đúng một lần.

## Ví dụ 1: Hai box lệch nhau một đơn vị có IoU bao nhiêu?

Box ghi theo dạng `xyxy`: mép trái, mép trên, mép phải, mép dưới. Trục x hướng sang phải, trục y hướng xuống dưới. Diện tích tính bằng rộng nhân cao, không cộng thêm 1. Box A là `[0, 0, 4, 4]`, box B là `[1, 1, 5, 5]`. Mỗi box có diện tích 16, phần giao là hình vuông 3×3, diện tích 9. IoU bằng diện tích giao chia diện tích hợp.

In [ ]:
import numpy as np

def iou(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if not (np.all(a[2:] > a[:2]) and np.all(b[2:] > b[:2])):
        raise ValueError("Box phải có rộng và cao dương")
    intersection = np.maximum(0, np.minimum(a[2:], b[2:]) - np.maximum(a[:2], b[:2])).prod()
    area_a, area_b = (a[2:] - a[:2]).prod(), (b[2:] - b[:2]).prod()
    return float(intersection / (area_a + area_b - intersection))

a, b = [0, 0, 4, 4], [1, 1, 5, 5]
print("IoU lệch:", round(iou(a, b), 4))
print("IoU trùng:", iou(a, a))
print("IoU chạm cạnh:", iou(a, [4, 0, 8, 4]))
assert np.isclose(iou(a, b), 9 / 23)
assert iou(a, b) == iou(b, a)
try:
    iou([2, 0, 1, 4], a)
except ValueError:
    print("Box đảo hai biên: bị từ chối")
else:
    raise AssertionError("Chưa bắt box sai")

IoU lệch: 0.3913
IoU trùng: 1.0
IoU chạm cạnh: 0.0
Box đảo hai biên: bị từ chối


Phần hợp là $16+16-9=23$, nên IoU bằng $9/23\approx0.39$. Hai box chỉ lệch nhau một đơn vị mỗi chiều mà IoU đã dưới 0.5, ngưỡng ghép thường dùng. IoU nhạy với độ lệch hơn ta tưởng, nhất là với box nhỏ.

Hai box chỉ chạm cạnh thì giao bằng 0, nên IoU bằng 0. Hàm `np.maximum(0, ...)` giữ cho độ rộng giao không bị âm. Một box có mép phải nằm bên trái mép trái là box hỏng, và bị từ chối ngay, thay vì lặng lẽ tạo ra diện tích âm.

## Ví dụ 2: NMS theo từng lớp giữ được vật thể khác lớp

Có bốn box ứng viên. Hai box đầu cùng lớp 0 và gần trùng nhau. Box thứ ba cũng lớp 0 nhưng nằm xa. Box thứ tư thuộc lớp 1, nằm đúng chỗ box đầu. NMS lần lượt giữ box có điểm cao nhất, rồi loại các box còn lại chồng lên nó với IoU lớn hơn 0.5. Ta so hai cách: xét riêng từng lớp, và trộn mọi lớp chung.

In [ ]:
import numpy as np

boxes = np.array([[0, 0, 4, 4], [0.5, 0.5, 4.5, 4.5],
                  [6, 0, 10, 4], [0, 0, 4, 4]], dtype=float)
scores = np.array([0.9, 0.8, 0.7, 0.6])
classes = np.array([0, 0, 0, 1])

def overlap(a, b):
    intersection = np.maximum(0, np.minimum(a[2:], b[2:]) - np.maximum(a[:2], b[:2])).prod()
    return intersection / ((a[2:] - a[:2]).prod() + (b[2:] - b[:2]).prod() - intersection)

def nms(boxes, scores, classes, threshold, per_class=True):
    pending = list(np.argsort(-scores, kind="stable"))
    kept = []
    while pending:
        best = pending.pop(0)
        kept.append(int(best))
        pending = [j for j in pending
                   if (per_class and classes[j] != classes[best])
                   or overlap(boxes[best], boxes[j]) <= threshold]
    return kept

print("Theo lớp:", nms(boxes, scores, classes, 0.5))
print("Trộn lớp:", nms(boxes, scores, classes, 0.5, per_class=False))
print("Ngưỡng 0.9:", nms(boxes, scores, classes, 0.9))
print("Không ứng viên:", nms(boxes[:0], scores[:0], classes[:0], 0.5))
assert nms(boxes, scores, classes, 0.5) == [0, 2, 3]
assert nms(boxes, scores, classes, 0.5, False) == [0, 2]

Theo lớp: [0, 2, 3]
Trộn lớp: [0, 2]
Ngưỡng 0.9: [0, 1, 2, 3]
Không ứng viên: []


Box 1 bị loại vì gần trùng box 0, cùng lớp mà điểm thấp hơn. Box 2 nằm xa nên được giữ. Box 3 là trường hợp đáng chú ý. Xét theo lớp, nó thuộc lớp 1, không cạnh tranh với box 0 lớp 0, nên được giữ. Trộn lớp chung, nó bị loại vì trùng chỗ box 0, dù có thể đó là một vật thể thật thuộc lớp khác, ví dụ một người đứng trước xe.

Nâng ngưỡng lên 0.9 thì box 1 không bị loại nữa, và bản trùng lặp lọt vào kết quả. Có ba ngưỡng khác nhau, rất dễ lẫn: ngưỡng IoU của NMS, ngưỡng điểm tin cậy để lọc box, và ngưỡng IoU dùng lúc chấm. Danh sách rỗng thì NMS trả về rỗng, không báo lỗi.

## Ví dụ 3: Box trùng lặp không được tính đúng lần thứ hai

Một ảnh có hai vật thể cùng lớp, A và B. Model đưa ra ba box, đã xếp theo điểm giảm dần: box ở A, thêm một box nữa ở A, rồi box ở B. Luật chấm ở đây là: box được tính đúng nếu IoU với một vật thể chưa được ghép đạt từ 0.5, và mỗi vật thể chỉ được ghép một lần. Bản này đơn giản hóa so với cách chấm COCO đầy đủ.

In [ ]:
import numpy as np

targets = np.array([[0, 0, 4, 4], [6, 0, 10, 4]], dtype=float)
predictions = targets[[0, 0, 1]]

def overlap(a, b):
    intersection = np.maximum(0, np.minimum(a[2:], b[2:]) - np.maximum(a[:2], b[:2])).prod()
    return intersection / ((a[2:] - a[:2]).prod() + (b[2:] - b[:2]).prod() - intersection)

matched, flags = set(), []
for box in predictions:
    candidates = [j for j in range(len(targets)) if j not in matched]
    best = max(candidates, key=lambda j: overlap(box, targets[j]), default=None)
    correct = best is not None and overlap(box, targets[best]) >= 0.5
    flags.append(int(correct))
    if correct:
        matched.add(best)
tp = np.cumsum(flags)
precision = tp / np.arange(1, len(flags) + 1)
recall = tp / len(targets)
# AP kiểu all-points: nâng precision thành đường bao không tăng, rồi cộng diện tích theo từng đoạn recall.
p = np.r_[0., precision, 0.]
r = np.r_[0., recall, 1.]
envelope = np.maximum.accumulate(p[::-1])[::-1]
ap = float(np.sum(np.diff(r) * envelope[1:]))
print("TP:", flags)
print("Precision:", precision.round(4).tolist())
print("Recall:", recall.tolist())
print("AP all-points toy:", round(ap, 4))
assert flags == [1, 0, 1]
assert np.isclose(ap, 5 / 6)

TP: [1, 0, 1]
Precision: [1.0, 0.5, 0.6667]
Recall: [0.5, 0.5, 1.0]
AP all-points toy: 0.8333


Box đầu ghép được với A, là một lần đúng. Box thứ hai cũng nằm đúng ở A, nhưng A đã được ghép rồi, nên nó bị tính là sai. Box thứ ba ghép với B. Nếu không có luật “mỗi vật thể một lần”, model chỉ cần lặp box là tăng được điểm.

Đọc precision và recall theo từng box. Sau box đầu: precision 1, recall 0.5. Sau box thứ hai: precision tụt còn 0.5, recall đứng yên. Sau box thứ ba: recall lên 1, precision là 2/3. AP là diện tích dưới đường precision theo recall, ở đây bằng $0.5\cdot1+0.5\cdot2/3=5/6$. Con số này chỉ tính trên ba box tự đặt, để kiểm cách chấm, không phải điểm của một model thật.